# 02 Baselines

**Goal:** build the train, validation and test split from the EDA, set a rule based baseline, and fit a first logistic regression.

Decisions carried over from `01_eda.ipynb`:
- Split inside 1 to 10 Sep 2022, with the 11 to 18 Sep tail held apart.
- `self_transfer` is a feature. Payment format and payment currency are ablated.
- Metrics are PR-AUC and recall at a fixed false positive rate.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../data")

TX_COLS = [
    "timestamp", "from_bank", "from_account", "to_bank", "to_account",
    "amount_received", "receiving_currency", "amount_paid",
    "payment_currency", "payment_format", "is_laundering",
]
TX_DTYPES = {
    "from_bank": "int32", "from_account": "category",
    "to_bank": "int32", "to_account": "category",
    "amount_received": "float64", "receiving_currency": "category",
    "amount_paid": "float64", "payment_currency": "category",
    "payment_format": "category", "is_laundering": "int8",
}


def load_transactions(path):
    tx = pd.read_csv(path, header=0, names=TX_COLS, dtype=TX_DTYPES)
    tx["timestamp"] = pd.to_datetime(tx["timestamp"], format="%Y/%m/%d %H:%M")
    tx["date"] = tx["timestamp"].dt.normalize()
    tx["self_transfer"] = (
        (tx["from_account"].astype(str) == tx["to_account"].astype(str))
        & (tx["from_bank"] == tx["to_bank"])
    )
    return tx


tx = load_transactions(DATA / "HI-Small_Trans.csv")
print(f"rows: {len(tx):,}")

rows: 5,078,345


In [2]:
SPLITS = {
    "train": ("2022-09-01", "2022-09-06"),
    "validation": ("2022-09-07", "2022-09-08"),
    "test": ("2022-09-09", "2022-09-10"),
}


def split_by_date(tx, splits=SPLITS):
    return {
        name: tx[(tx["date"] >= start) & (tx["date"] <= end)]
        for name, (start, end) in splits.items()
    }


parts = split_by_date(tx)
train, val, test = parts["train"], parts["validation"], parts["test"]

for name, part in parts.items():
    print(f"{name:<11} rows: {len(part):>9,}   laundering: {part['is_laundering'].sum():>5,}")

train       rows: 3,248,921   laundering: 2,530
validation  rows:   965,524   laundering: 1,036
test        rows:   862,792   laundering:   956


## Evaluation

Every model is scored the same way on the validation set:
- **PR-AUC** (average precision), the main metric at this class balance.
- **Recall at a fixed false positive rate** of 0.1% and 1%, which answers "how much laundering is caught if analysts can only review this many alerts".

A model that ranks at random has a PR-AUC equal to the laundering rate, about 0.001.

In [3]:
from sklearn.metrics import average_precision_score, roc_curve


def recall_at_fpr(y_true, scores, max_fpr):
    fpr, tpr, _ = roc_curve(y_true, scores)
    return float(tpr[fpr <= max_fpr].max())


def evaluate(y_true, scores, name, fprs=(0.001, 0.01)):
    result = {"model": name, "pr_auc": average_precision_score(y_true, scores)}
    for f in fprs:
        result[f"recall_at_{f:.1%}_fpr"] = recall_at_fpr(y_true, scores, f)
    return result


results = []
y_val = val["is_laundering"].to_numpy()
print(f"validation laundering rate: {y_val.mean():.4%}")

validation laundering rate: 0.1073%


In [4]:
ach_flag = (val["payment_format"] == "ACH").to_numpy().astype(int)

tp = int(((ach_flag == 1) & (y_val == 1)).sum())
fp = int(((ach_flag == 1) & (y_val == 0)).sum())
fn = int(((ach_flag == 0) & (y_val == 1)).sum())
tn = int(((ach_flag == 0) & (y_val == 0)).sum())

print(f"alerts raised: {tp + fp:,} of {len(val):,} transactions")
print(f"precision: {tp / (tp + fp):.2%}")
print(f"recall: {tp / (tp + fn):.2%}")
print(f"false positive rate: {fp / (fp + tn):.2%}")

results.append(evaluate(y_val, ach_flag, "ACH rule"))
pd.DataFrame(results)

alerts raised: 116,910 of 965,524 transactions
precision: 0.76%
recall: 85.91%
false positive rate: 12.03%


,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr
0,ACH rule,0.006691,0.0,0.0


## Features and logistic regression

First model: logistic regression on transaction level features only. It is fitted twice, without and with the two shortcut columns (payment format and payment currency), to see how much of the score depends on them.

In [5]:
currency_median = (
    train.groupby("payment_currency", observed=True)["amount_paid"].median().to_dict()
)


def add_features(df, currency_median):
    out = pd.DataFrame(index=df.index)
    out["log_amount"] = np.log10(df["amount_paid"].clip(lower=0.01))
    typical = df["payment_currency"].astype(str).map(currency_median)
    out["amount_vs_currency"] = out["log_amount"] - np.log10(typical)
    hour = df["timestamp"].dt.hour
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    out["self_transfer"] = df["self_transfer"].astype(int)
    out["payment_format"] = df["payment_format"].astype(str)
    out["payment_currency"] = df["payment_currency"].astype(str)
    return out


X_train = add_features(train, currency_median)
X_val = add_features(val, currency_median)
y_train = train["is_laundering"].to_numpy()

X_train.head()

,log_amount,amount_vs_currency,hour_sin,hour_cos,self_transfer,payment_format,payment_currency
0,3.567889,0.573775,0.0,1.0,1,Reinvestment,US Dollar
1,-2.000000,-4.994115,0.0,1.0,0,Cheque,US Dollar
2,4.166595,1.172480,0.0,1.0,1,Reinvestment,US Dollar
3,3.448238,0.454123,0.0,1.0,1,Reinvestment,US Dollar
4,4.564464,1.570350,0.0,1.0,1,Reinvestment,US Dollar


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = ["log_amount", "amount_vs_currency", "hour_sin", "hour_cos", "self_transfer"]
SHORTCUTS = ["payment_format", "payment_currency"]


def fit_logreg(X, y, categorical):
    steps = [("num", StandardScaler(), NUMERIC)]
    if categorical:
        steps.append(("cat", OneHotEncoder(handle_unknown="ignore"), categorical))
    model = make_pipeline(
        ColumnTransformer(steps),
        LogisticRegression(class_weight="balanced", max_iter=500),
    )
    return model.fit(X, y)


models = {}
for name, categorical in [
    ("LogReg, no shortcut columns", []),
    ("LogReg, with format and currency", SHORTCUTS),
]:
    models[name] = fit_logreg(X_train, y_train, categorical)
    scores = models[name].predict_proba(X_val)[:, 1]
    results.append(evaluate(y_val, scores, name))

pd.DataFrame(results)

,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr
0,ACH rule,0.006691,0.000000,0.000000
1,"LogReg, no shortcut columns",0.002210,0.001931,0.018340
2,"LogReg, with format and currency",0.016349,0.031853,0.131274


## Gradient boosting

Same features as the logistic regression, with a non-linear model. This tests whether the weak result without shortcut columns was a limit of the linear model or of the features.

In [7]:
import lightgbm as lgb

CATEGORIES = {c: sorted(X_train[c].unique()) for c in SHORTCUTS}


def gbm_frame(X, use_shortcuts):
    cols = NUMERIC + (SHORTCUTS if use_shortcuts else [])
    out = X[cols].copy()
    for c in SHORTCUTS:
        if c in out:
            out[c] = pd.Categorical(out[c], categories=CATEGORIES[c])
    return out


def fit_gbm(X, y, use_shortcuts):
    model = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=200,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1,
    )
    return model.fit(gbm_frame(X, use_shortcuts), y)


for name, use_shortcuts in [
    ("LightGBM, no shortcut columns", False),
    ("LightGBM, with format and currency", True),
]:
    models[name] = fit_gbm(X_train, y_train, use_shortcuts)
    scores = models[name].predict_proba(gbm_frame(X_val, use_shortcuts))[:, 1]
    results.append(evaluate(y_val, scores, name))

pd.DataFrame(results)

,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr
0,ACH rule,0.006691,0.000000,0.000000
1,"LogReg, no shortcut columns",0.002210,0.001931,0.018340
2,"LogReg, with format and currency",0.016349,0.031853,0.131274
3,"LightGBM, no shortcut columns",0.005784,0.016409,0.108108
4,"LightGBM, with format and currency",0.062708,0.139961,0.460425


**Finding:** With the same features, LightGBM clearly beats logistic regression. Without the shortcut columns, recall at a 1% false positive rate rises from 19 to 112 of 1,036 laundering transactions, which supports the view that the linear model could not represent the amount band seen in the EDA. PR-AUC without shortcut columns is still only 0.0058, just under the ACH rule (0.0067), so transaction level features remain the main limit. With payment format and payment currency, LightGBM reaches a PR-AUC of 0.0627 and catches 477 of 1,036 laundering transactions (46%) at a 1% false positive rate, about one true alert in 21, against one in 131 for the ACH rule. The shortcut columns multiply PR-AUC by about eleven, so the model still depends heavily on them. The next step adds features describing each account's activity across transactions.